# Forced-span probe: manual validation

Numbered 5 because `4_coverage_gap_inspection.ipynb` already exists.

## What the probe did

`given` and `given that` are DiMLex connectives that discopy never enumerates
as candidates, so its sense classifier never sees them. This probe hands the
exact spans to the **unchanged** classifier, bypassing **only** the
candidate-enumeration step, and records what comes back.

Nothing else changed: same weights, same `get_bert_features`, same
`used_context`, same argmax, and the full document embedded so the +/-1 token
window sees the same neighbours it would in a real run. No hybrid is built or
activated, and the standard discopy outputs are untouched.

## What acceptance does and does not mean

**A non-`NoSense` label is not success.** It only means the classifier returned
something. Three separate things have to hold before a hybrid is worth
building, and only manual review can establish them:

1. the span is genuinely a PDTB-style Explicit connective;
2. the top-level sense is correct;
3. the finer sense is reasonable.

Classifier output is not treated as gold anywhere in this notebook.

## Annotation criterion

> Given the PDTB-style Explicit-relation criterion that discopy implements, is
> this forced span a valid Explicit discourse connective - and if so, is the
> classifier's sense appropriate?

**Not** the broader "does this expression convey some discourse relation?".
A construction can express causality and still not be a PDTB Explicit
connective - which is precisely why these forms are outside the inventory.

In [1]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root {repo_name!r} above {Path.cwd()}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.validation.forced_span_review import (
    ForcedSpanReviewApp, ForcedSpanStore, MANUAL_COLUMNS,
    progress_summary, render_forced_case,
)
from src.justification_analysis.comparison import forced_span_summary as fs

PROBE_DIR = (
    REPO_ROOT / "analysis" / "cross_model" / "base" / "voting" / "prompt_v4"
    / "justification_analysis" / "discourse_parser" / "forced_span_probe"
)

PREDICTIONS_PATH = PROBE_DIR / "forced_span_predictions.csv"
SAMPLE_PATH = PROBE_DIR / "forced_span_validation_sample_30.csv"
COMPLETED_PATH = PROBE_DIR / "forced_span_validation_completed.csv"

SEED = 20260826

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("predictions:", PREDICTIONS_PATH.name)
print("sample     :", SAMPLE_PATH.name)
print("answers    :", COMPLETED_PATH.name)

predictions: forced_span_predictions.csv
sample     : forced_span_validation_sample_30.csv
answers    : forced_span_validation_completed.csv


## 1. Corpus-level probe results

Descriptive summary of every forced span. This is the classifier's raw
behaviour, not a validated result.

In [2]:
probe = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")
probe = fs.add_confidence_band(probe)

print(f"forced spans scored: {len(probe):,}")
print(f"accepted           : {int(probe['accepted'].sum()):,} "
      f"({100 * probe['accepted'].mean():.1f}%)")
print(f"NoSense            : {int(probe['is_nosense'].sum()):,} "
      f"({100 * probe['is_nosense'].mean():.1f}%)")

print("\nBy form:")
display(fs.acceptance_by(probe, ["form"]))

print("\nBy model and decoding:")
display(fs.acceptance_by(probe, ["model", "decoding_group"]))

forced spans scored: 531
accepted           : 103 (19.4%)
NoSense            : 428 (80.6%)

By form:


,form,n_forced,n_accepted,n_nosense,acceptance_rate_pct,mean_confidence
0,given,477,50,427,10.5,0.943629
1,given that,54,53,1,98.1,0.475540



By model and decoding:


,model,decoding_group,n_forced,n_accepted,n_nosense,acceptance_rate_pct,mean_confidence
0,Gemma 4 2B,Greedy,39,14,25,35.9,0.827804
1,Gemma 4 2B,Stochastic,128,32,96,25.0,0.875451
2,Gemma 4 31B,Greedy,24,3,21,12.5,0.919023
3,Gemma 4 31B,Stochastic,73,21,52,28.8,0.838996
4,Gemma 4 4B,Greedy,75,9,66,12.0,0.929131
5,Gemma 4 4B,Stochastic,192,24,168,12.5,0.929479


In [3]:
print("Predicted sense distribution - given:")
display(fs.sense_distribution(probe, "given"))

print("\nPredicted sense distribution - given that:")
display(fs.sense_distribution(probe, "given that"))

print("\nAmong ACCEPTED spans only, where does the classifier put them?")
print("(the manual inspection judged every one of these forms Contingency)")
display(fs.accepted_top_level_distribution(probe))

print("\nConfidence by outcome:")
display(fs.confidence_summary(probe))

Predicted sense distribution - given:


,predicted_sense,n,pct,top_level
0,NoSense,427,89.5,NoSense
1,Contingency.Cause,50,10.5,Contingency



Predicted sense distribution - given that:


,predicted_sense,n,pct,top_level
0,Contingency.Cause,41,75.9,Contingency
1,Temporal.Asynchronous,12,22.2,Temporal
2,NoSense,1,1.9,NoSense



Among ACCEPTED spans only, where does the classifier put them?
(the manual inspection judged every one of these forms Contingency)


,form,n_accepted,Comparison,Contingency,Expansion,Temporal,pct_Contingency
0,given,50,0,50,0,0,100.0
1,given that,53,0,41,0,12,77.4
2,ALL,103,0,91,0,12,88.3



Confidence by outcome:


accepted,NoSense,accepted
confidence_band,,
<0.50,1,49
0.50-0.75,0,52
0.75-0.90,9,1
>=0.90,418,1


## 2. Reproducible 30-case sample

Fixed seed, stratified across form, prediction (accepted vs `NoSense`), model,
decoding and confidence band, so the review covers the classifier's behaviour
rather than only its most common output. The sampled rows are written to CSV so
the sample is reproducible independently of this notebook.

In [4]:
# Quotas by (form x prediction). The population is 81% NoSense, but a
# proportional sample would leave too few accepted cases to judge whether the
# acceptances are sound - which is half the question. Both arms are therefore
# represented deliberately, capped at what exists (`given that` has only one
# NoSense case in the whole probe). This is a coverage design, not a
# probability sample, so it supports counts and never a rate.
QUOTAS = {
    ("given", True): 10,
    ("given", False): 10,
    ("given that", True): 9,
    ("given that", False): 1,
}


def build_sample(frame, seed=SEED, quotas=QUOTAS):
    """Fixed-seed sample, quota per (form, prediction), spread within cell."""
    rng = np.random.RandomState(seed)
    frame = frame.copy()
    picked = []

    for (form, accepted), quota in quotas.items():
        cell = frame.loc[frame["form"].eq(form) & frame["accepted"].eq(accepted)]
        if not len(cell):
            continue
        chosen = []
        # Spread within the cell across model, decoding and confidence first.
        for column in ("model", "decoding_group", "confidence_band"):
            for value in cell[column].dropna().unique():
                if len(chosen) >= quota:
                    break
                pool = cell.loc[cell[column].eq(value) & ~cell.index.isin(chosen)]
                if len(pool):
                    chosen.append(pool.sample(1, random_state=rng).index[0])
        leftover = cell.loc[~cell.index.isin(chosen)]
        if len(chosen) < quota and len(leftover):
            chosen += list(leftover.sample(min(quota - len(chosen), len(leftover)),
                                           random_state=rng).index)
        picked += chosen[:quota]

    return frame.loc[picked].reset_index(drop=True)


sample = build_sample(probe)

for column in MANUAL_COLUMNS:
    sample[column] = ""

SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"sampled: {len(sample)} cases (seed {SEED})")
print(f"saved  -> {SAMPLE_PATH}")

print("\nstratum coverage:")
display(sample.groupby(["form", "accepted"], observed=True)
        .size().rename("n").reset_index())
display(sample.groupby(["model", "decoding_group"], observed=True)
        .size().rename("n").reset_index())
display(sample["confidence_band"].value_counts()
        .rename_axis("confidence_band").reset_index(name="n"))

sampled: 30 cases (seed 20260826)
saved  -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\forced_span_probe\forced_span_validation_sample_30.csv

stratum coverage:


,form,accepted,n
0,given,False,10
1,given,True,10
2,given that,False,1
3,given that,True,9


,model,decoding_group,n
0,Gemma 4 2B,Greedy,4
1,Gemma 4 2B,Stochastic,6
2,Gemma 4 31B,Greedy,2
3,Gemma 4 31B,Stochastic,6
4,Gemma 4 4B,Greedy,1
5,Gemma 4 4B,Stochastic,11


,confidence_band,n
0,0.50-0.75,10
1,>=0.90,10
2,<0.50,8
3,0.75-0.90,2


## 3. Review

One case at a time. The span is highlighted from its exact character offsets,
and the classifier's prediction and confidence are shown - this is targeted
inspection, not blinded annotation.

For each case:

1. **Is this a valid PDTB-style Explicit connective?** (the narrow criterion)
2. **Is the top-level sense correct?** — `n/a` if it is not a connective, or if
   the classifier said `NoSense`
3. **Is the full sense reasonable?**
4. **Expected top-level category**, if it is a connective
5. **Expected full sense** and **notes**, both optional

Every control saves immediately to `forced_span_validation_completed.csv`.

In [5]:
cases = sample.to_dict("records")

# Locate the span inside the sentence so the highlight is exact.
import re
for case in cases:
    sentence = str(case.get("sentence_text", ""))
    match = re.search(rf"(?<!\w){re.escape(str(case['marker']))}(?!\w)",
                      sentence, re.I)
    case["sent_start"] = match.start() if match else -1
    case["sent_end"] = match.end() if match else -1

store = ForcedSpanStore(COMPLETED_PATH, cases)

assert len(cases) == 30, f"expected 30 cases, got {len(cases)}"
assert len({c["probe_id"] for c in cases}) == len(cases), "duplicate probe_id"

print(f"cases loaded    : {len(cases)}")
print(f"already answered: {store.n_answered()} / {len(cases)}")

app = ForcedSpanReviewApp(cases, store)
display(app.ui)

cases loaded    : 30
already answered: 0 / 30


## 4. Progress and results

Re-run to check progress. Results appear only once all 30 are answered - raw
counts, no rate, no accuracy metric. The sample is stratified for coverage, not
drawn proportionally, so it does not support a population estimate.

In [6]:
store.save()

answers = store.to_frame()
n_answered = store.n_answered()

print(f"answered: {n_answered} / {len(cases)}")
print(f"saved -> {COMPLETED_PATH}")
display(progress_summary(cases, store))

if n_answered < len(cases):
    remaining = [int(c["probe_id"]) for c in cases
                 if not store.is_answered(c)]
    print(f"\nstill to review (probe_id): {remaining}")
else:
    is_conn = answers["manual_is_explicit_connective"].str.lower().eq("yes")
    print(f"\njudged valid Explicit connectives: "
          f"{int(is_conn.sum())}/{len(answers)}")

    print("\nby form and classifier prediction:")
    display(
        answers.assign(
            valid=is_conn,
            prediction=np.where(answers["accepted"].astype(str).str.lower()
                                .isin(["true", "1"]), "accepted", "NoSense"),
        )
        .groupby(["form", "prediction"], observed=True)
        .agg(n=("valid", "size"), judged_valid=("valid", "sum"))
    )

    scored = answers.loc[
        answers["manual_top_level_correct"].str.lower().isin(["yes", "no"])
    ]
    if len(scored):
        print("\ntop-level sense correct (where scoreable): "
              f"{int(scored['manual_top_level_correct'].str.lower().eq('yes').sum())}"
              f"/{len(scored)}")
    scored_full = answers.loc[
        answers["manual_full_sense_correct"].str.lower().isin(["yes", "no"])
    ]
    if len(scored_full):
        print("full sense reasonable (where scoreable): "
              f"{int(scored_full['manual_full_sense_correct'].str.lower().eq('yes').sum())}"
              f"/{len(scored_full)}")

    print("\nRaw counts only. Stratified for coverage, so no rate and no "
          "population estimate follows from these numbers.")

answered: 30 / 30
saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\forced_span_probe\forced_span_validation_completed.csv


,form,prediction,n,answered,remaining
0,given,NoSense,10,10,0
1,given,accepted,10,10,0
2,given that,NoSense,1,1,0
3,given that,accepted,9,9,0



judged valid Explicit connectives: 20/30

by form and classifier prediction:


n  judged_valid
form       prediction                  
given      NoSense     10             0
           accepted    10            10
given that NoSense      1             1
           accepted     9             9


top-level sense correct (where scoreable): 20/20
full sense reasonable (where scoreable): 20/20

Raw counts only. Stratified for coverage, so no rate and no population estimate follows from these numbers.


### Static view (optional)

In [7]:
from IPython.display import HTML, display as _display

CASE_TO_SHOW = 1   # 1-30

_display(HTML(render_forced_case(
    cases[CASE_TO_SHOW - 1], CASE_TO_SHOW, len(cases)
)))

## 5. What the outcome means

**Outcome A** - forced spans mostly `NoSense`, or accepted with wrong/unstable
senses: a forced-span hybrid is **not supported**. Standard discopy stays the
main analysis, and the coverage gap remains a documented limitation backed by
the sensitivity tables (`12_contingency_sensitivity.csv`,
`14_four_class_profiles_sensitivity.csv`).

**Outcome B** - forced spans consistently recognised with sensible Contingency
senses: a minimal hybrid is **technically plausible**. It would require
supplying DiMLex spans for these two forms to the classifier at parse time and
merging the results into the occurrence table, with the sense taken from the
classifier rather than from DiMLex. **Not to be implemented without explicit
approval.**

Either way, nothing in the production pipeline changes on the basis of this
notebook alone.